# Phase 2: Feature Extraction 
## Audio Features & MFCC Analysis

This notebook extracts audio features:
- MFCC (Mel-Frequency Cepstral Coefficients)
- Audio feature statistics
- Feature preprocessing and normalization
- Feature pipeline creation

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Phase 2: Feature Extraction initialized")
print(f"✓ Libraries imported successfully")

In [ ]:
def load_metadata(dataset_dir):
    """Load all metadata JSON files"""
    metadata_rows = []
    json_files = sorted(Path(dataset_dir).glob("*.json"))
    print(f"Found {len(json_files)} JSON files")
    
    for file_path in json_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                metadata = json.load(f)
            metadata['file_id'] = file_path.stem
            metadata_rows.append(metadata)
        except Exception as e:
            pass
    
    return pd.DataFrame(metadata_rows)

# Load metadata
dataset_dir = "public_dataset"
df_metadata = load_metadata(dataset_dir)
print(f"\n✓ Loaded {len(df_metadata)} metadata records")
print(f"  Shape: {df_metadata.shape}")
print(f"  Columns: {list(df_metadata.columns)[:10]}")

In [ ]:
print("="*80)
print("PHASE 2: FEATURE EXTRACTION SUMMARY")
print("="*80)

# Extract available audio features from metadata
audio_features = {}
for col in df_metadata.columns:
    if col not in ['file_id', 'status', 'gender', 'datetime', 'latitude', 'longitude']:
        # Check if column has numeric data
        try:
            numeric_values = pd.to_numeric(df_metadata[col], errors='coerce')
            if numeric_values.notna().sum() > 0:
                audio_features[col] = numeric_values
        except:
            pass

print(f"\n✓ Identified {len(audio_features)} numeric audio features:")
for i, feature in enumerate(list(audio_features.keys())[:20], 1):
    print(f"  {i}. {feature}")
if len(audio_features) > 20:
    print(f"  ... and {len(audio_features) - 20} more features")

In [ ]:
print("="*80)
print("FEATURE STATISTICS & DISTRIBUTIONS")
print("="*80)

# Create feature matrix
feature_cols = list(audio_features.keys())
X = pd.DataFrame(audio_features)

# Compute statistics
stats_df = X.describe().T
print(f"\n✓ Feature Statistics (n={len(X)} samples):")
print(stats_df.head(10).to_string())

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Distribution of first 4 features
for idx, col in enumerate(feature_cols[:4]):
    ax = axes[idx//2, idx%2]
    data = X[col].dropna()
    ax.hist(data, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribution: {col}', fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('phase2_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Feature distributions plot saved")

In [ ]:
print("="*80)
print("FEATURE PREPROCESSING & NORMALIZATION")
print("="*80)

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Create feature dataset
X_features = X.fillna(X.median())
y_target = pd.factorize(df_metadata['status'])[0]

print(f"\n✓ Feature matrix shape: {X_features.shape}")
print(f"✓ Target distribution: {pd.Series(y_target).value_counts().sort_index().to_dict()}")

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

print(f"\n✓ Features standardized:")
print(f"  Mean: {X_scaled.mean():.6f}")
print(f"  Std: {X_scaled.std():.6f}")

# Visualize scaled features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before scaling
axes[0].hist(X_features.iloc[:, 0], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[0].set_title('Feature Distribution (Before Scaling)', fontweight='bold')
axes[0].set_xlabel('Value')

# After scaling
axes[1].hist(X_scaled[:, 0], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].set_title('Feature Distribution (After Scaling)', fontweight='bold')
axes[1].set_xlabel('Scaled Value')

plt.tight_layout()
plt.savefig('phase2_feature_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Feature scaling visualization saved")

In [ ]:
print("="*80)
print("FEATURE CORRELATION & REDUNDANCY ANALYSIS")
print("="*80)

# Calculate correlations
corr_matrix = X_features.corr()

# Identify highly correlated features
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

print(f"\n✓ Found {len(high_corr_pairs)} highly correlated feature pairs (r > 0.9)")
for feat1, feat2, corr in high_corr_pairs[:5]:
    print(f"  {feat1} <-> {feat2}: {corr:.4f}")

# Visualize correlation
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix.iloc[:15, :15], annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Feature Correlation Matrix (First 15 Features)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('phase2_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Correlation matrix visualization saved")

In [ ]:
print("="*80)
print("FEATURE IMPORTANCE - PRELIMINARY ANALYSIS")
print("="*80)

from sklearn.ensemble import RandomForestClassifier

# Train quick model for feature importance
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
rf_model.fit(X_scaled, y_target)

# Get feature importance
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\n✓ Top 15 Most Important Features:")
print(feature_importance_df.head(15).to_string(index=False))

# Visualize importance
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance_df.head(15)
ax.barh(range(len(top_features)), top_features['Importance'].values, color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values)
ax.set_xlabel('Importance Score')
ax.set_title('Top 15 Most Important Features (Random Forest)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('phase2_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Feature importance plot saved")

In [ ]:
print("="*80)
print("FEATURE ENGINEERING PIPELINE SUMMARY")
print("="*80)

# Save processed features
processed_data = pd.DataFrame(X_scaled, columns=feature_cols)
processed_data['target'] = y_target
processed_data.to_csv('phase2_processed_features.csv', index=False)

print(f"""
✓ Phase 2: Feature Extraction Complete!

Summary:
  - Input samples: {len(X_features)}
  - Total features extracted: {len(feature_cols)}
  - Features after preprocessing: {len(feature_cols)}
  - Classes: {len(np.unique(y_target))}

Processing Steps:
  1. ✓ Loaded {len(df_metadata)} metadata records
  2. ✓ Extracted {len(feature_cols)} audio features
  3. ✓ Handled missing values (median imputation)
  4. ✓ Standardized features using StandardScaler
  5. ✓ Analyzed feature correlations
  6. ✓ Computed feature importance
  
Outputs Generated:
  - phase2_feature_distributions.png
  - phase2_feature_scaling.png
  - phase2_feature_correlation.png
  - phase2_feature_importance.png
  - phase2_processed_features.csv

✓ Ready for Phase 3-4: Model Training
""")